# RNNs and Sequence Models

Text, time series, audio, DNA — data where **order matters**. A regular neural network treats inputs as independent: shuffling the pixels of an image before feeding it to a dense net wouldn't help, but it wouldn't *break the architecture*. For sequences, order IS the signal.

"The cat sat on the" → what comes next? You need context from all previous words.

**RNNs** solve this by maintaining a **hidden state** — a memory that carries information from previous steps.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

---
## 1. The RNN Concept — A Loop Through Time

An RNN processes one element at a time, updating its hidden state at each step:

```
Unrolled RNN:

  x1 → [RNN] → h1 → [RNN] → h2 → [RNN] → h3 → output
         ↑              ↑              ↑
        h0             h1             h2
        (init)     (carries info   (carries info
                    from x1)        from x1, x2)
```

At each step: `h_t = tanh(W_hh · h_{t-1} + W_xh · x_t + b)`

The same weights (`W_hh`, `W_xh`) are reused at every time step — **weight sharing across time**.

In [ ]:
class SimpleRNN:
    def __init__(self, input_size, hidden_size):
        scale = np.sqrt(1.0 / hidden_size)
        self.W_xh = np.random.randn(input_size, hidden_size) * scale
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale
        self.b_h = np.zeros(hidden_size)
        self.hidden_size = hidden_size
    
    def forward(self, sequence):
        h = np.zeros(self.hidden_size)
        states = [h.copy()]
        
        for x_t in sequence:
            h = np.tanh(x_t @ self.W_xh + h @ self.W_hh + self.b_h)
            states.append(h.copy())
        
        return h, states

rnn = SimpleRNN(input_size=4, hidden_size=8)
sequence = np.random.randn(5, 4)
final_h, all_states = rnn.forward(sequence)

print(f"Input sequence: {sequence.shape[0]} steps, {sequence.shape[1]} features each")
print(f"Hidden state dimension: {rnn.hidden_size}")
print(f"Final hidden state: [{', '.join(f'{v:.3f}' for v in final_h[:5])}...]")

fig, ax = plt.subplots(figsize=(10, 4))
states_arr = np.array(all_states)
im = ax.imshow(states_arr.T, cmap='RdBu', aspect='auto', interpolation='nearest')
ax.set_xlabel('Time Step')
ax.set_ylabel('Hidden Unit')
ax.set_title('Hidden State Evolution Over Time', fontweight='bold')
ax.set_xticks(range(len(all_states)))
ax.set_xticklabels(['h₀'] + [f'h{i+1}' for i in range(len(sequence))])
plt.colorbar(im, label='Activation')
plt.tight_layout()
plt.show()

---
## 2. The Vanishing Gradient Problem

During backpropagation through time, gradients are multiplied at each step. If the multiplier is < 1, gradients **shrink exponentially**. After 50-100 steps, the gradient from the last step barely reaches the first — the network "forgets" early inputs.

This is why basic RNNs fail on long sequences.

In [ ]:
seq_lengths = [5, 10, 20, 50, 100]
gradient_magnitude = []

for T in seq_lengths:
    grad_scale = 0.95
    total_gradient = grad_scale ** T
    gradient_magnitude.append(total_gradient)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

steps = np.arange(100)
for factor, label, color in [(0.5, 'γ=0.5 (vanishing)', '#e74c3c'),
                              (0.95, 'γ=0.95 (slow vanish)', '#f39c12'),
                              (1.0, 'γ=1.0 (ideal)', '#2ecc71'),
                              (1.05, 'γ=1.05 (exploding)', '#3498db')]:
    grads = factor ** steps
    ax1.plot(steps, grads, label=label, linewidth=2, color=color)

ax1.set_xlabel('Time Steps Back')
ax1.set_ylabel('Gradient Magnitude')
ax1.set_title('Gradient Flow Through Time', fontweight='bold')
ax1.set_ylim(-0.1, 5)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(range(len(seq_lengths)), gradient_magnitude, color='#e74c3c', alpha=0.8)
ax2.set_xticks(range(len(seq_lengths)))
ax2.set_xticklabels([str(s) for s in seq_lengths])
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Gradient at First Step')
ax2.set_title('Gradient Reaching First Step (γ=0.95)', fontweight='bold')
ax2.grid(True, alpha=0.3)
for i, v in enumerate(gradient_magnitude):
    ax2.text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()
print("At 100 steps with γ=0.95, gradient is 0.6% of original — essentially zero.")
print("The network can't learn dependencies that span more than ~20 steps.")

---
## 3. LSTM — Long Short-Term Memory

LSTMs solve vanishing gradients with **gates** that control information flow:

```
┌───────────────────────────────────────────┐
│                LSTM Cell                   │
│                                           │
│  ┌─────┐   ┌─────┐   ┌─────┐             │
│  │Forget│   │Input│   │Output│             │
│  │ Gate │   │Gate │   │ Gate │             │
│  └──┬──┘   └──┬──┘   └──┬──┘             │
│     │         │         │                 │
│     ▼         ▼         ▼                 │
│  c_{t-1} ──×──⊕──────────×──► c_t        │
│            │  │          │                 │
│            │  └── tanh ──┘                 │
│                     │                      │
│                     ▼                      │
│                   h_t ──► output           │
└───────────────────────────────────────────┘
```

- **Forget gate**: what to erase from cell state ("forget the subject when we see a period")
- **Input gate**: what new info to store ("remember the new subject")
- **Output gate**: what to expose as hidden state ("output verb tense info")

The cell state `c_t` can carry information unchanged for hundreds of steps (the gradient highway).

In [ ]:
lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=1, batch_first=True)

x = torch.randn(3, 5, 10)
output, (h_n, c_n) = lstm(x)

print(f"Input:             {x.shape}       (batch=3, seq_len=5, features=10)")
print(f"Output (all h's):  {output.shape}  (batch=3, seq_len=5, hidden=20)")
print(f"h_n (final h):     {h_n.shape}     (layers=1, batch=3, hidden=20)")
print(f"c_n (final cell):  {c_n.shape}     (layers=1, batch=3, hidden=20)")
print(f"\noutput[:, -1, :] == h_n[0]: {torch.allclose(output[:, -1, :], h_n[0])}")
print("The last output equals the final hidden state.")

In [ ]:
stacked_lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=3, batch_first=True, dropout=0.1)

x = torch.randn(2, 5, 10)
output, (h_n, c_n) = stacked_lstm(x)

print(f"Stacked LSTM (3 layers):")
print(f"  Output: {output.shape}")
print(f"  h_n:    {h_n.shape}  (one hidden state per layer)")
print(f"  c_n:    {c_n.shape}")
print(f"\n  Parameters: {sum(p.numel() for p in stacked_lstm.parameters()):,}")

---
## 4. Time Series Prediction — Sine Wave

A clean test: can the LSTM learn the pattern of a sine wave and predict the next value?

In [ ]:
t = np.linspace(0, 20 * np.pi, 1000)
signal = np.sin(t)

seq_length = 50
X_seq, y_seq = [], []
for i in range(len(signal) - seq_length):
    X_seq.append(signal[i:i+seq_length])
    y_seq.append(signal[i+seq_length])

X_seq = torch.FloatTensor(np.array(X_seq)).unsqueeze(-1)
y_seq = torch.FloatTensor(np.array(y_seq)).unsqueeze(-1)

split = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

print(f"Training sequences: {X_train.shape[0]}")
print(f"Test sequences:     {X_test.shape[0]}")
print(f"Sequence shape:     {X_train.shape}  (samples, timesteps, features)")

In [ ]:
class SinePredictor(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        output, (h_n, c_n) = self.lstm(x)
        return self.fc(h_n[-1])

torch.manual_seed(42)
model = SinePredictor(hidden_size=32).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

losses = []
for epoch in range(30):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(train_loader))
    if epoch % 5 == 0:
        print(f"  Epoch {epoch:2d} | Loss: {losses[-1]:.6f}")

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(X_test.to(device)).cpu().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(losses, color='#e74c3c', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(y_test.numpy()[:100], label='Actual', color='#3498db', linewidth=2)
ax2.plot(preds[:100], label='Predicted', color='#e74c3c', linewidth=2, linestyle='--')
ax2.set_xlabel('Time Step')
ax2.set_ylabel('Value')
ax2.set_title('Sine Wave Prediction', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

mse = np.mean((preds - y_test.numpy()) ** 2)
print(f"Test MSE: {mse:.6f}")

---
## 5. Text Generation — Character-Level LSTM

We train a character-level model: given a sequence of characters, predict the next one. By repeatedly predicting, we generate new text.

In [ ]:
text = """To be or not to be that is the question.
Whether tis nobler in the mind to suffer the slings and arrows of outrageous fortune,
or to take arms against a sea of troubles and by opposing end them.
To die to sleep no more and by a sleep to say we end the heartache
and the thousand natural shocks that flesh is heir to.
Tis a consummation devoutly to be wished.
To die to sleep to sleep perchance to dream."""

chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
vocab_size = len(chars)

print(f"Text length: {len(text)} characters")
print(f"Vocabulary: {vocab_size} unique characters")
print(f"Characters: {''.join(chars)}")

In [ ]:
seq_len = 30
encoded = [char_to_idx[c] for c in text]

X_chars, y_chars = [], []
for i in range(len(encoded) - seq_len):
    X_chars.append(encoded[i:i+seq_len])
    y_chars.append(encoded[i+seq_len])

X_chars = torch.LongTensor(X_chars)
y_chars = torch.LongTensor(y_chars)

print(f"Training sequences: {len(X_chars)}")
print(f"Example input:  {''.join(idx_to_char[i.item()] for i in X_chars[0])}")
print(f"Example target: {idx_to_char[y_chars[0].item()]}")

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        output, _ = self.lstm(x)
        return self.fc(output[:, -1, :])
    
    def generate(self, seed_text, length=100, temperature=0.8):
        self.eval()
        chars_generated = list(seed_text)
        input_seq = [char_to_idx.get(c, 0) for c in seed_text[-seq_len:]]
        
        with torch.no_grad():
            for _ in range(length):
                x = torch.LongTensor([input_seq[-seq_len:]]).to(device)
                logits = self(x) / temperature
                probs = torch.softmax(logits, dim=-1)
                next_idx = torch.multinomial(probs, 1).item()
                chars_generated.append(idx_to_char[next_idx])
                input_seq.append(next_idx)
        
        return ''.join(chars_generated)

torch.manual_seed(42)
char_model = CharLSTM(vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(char_model.parameters(), lr=0.005)

dataset = torch.utils.data.TensorDataset(X_chars, y_chars)
loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

In [ ]:
losses = []
for epoch in range(100):
    char_model.train()
    epoch_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        output = char_model(X_batch)
        loss = criterion(output, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(loader))
    if epoch % 20 == 0:
        print(f"  Epoch {epoch:3d} | Loss: {losses[-1]:.4f}")

plt.plot(losses, color='#9b59b6', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Character LSTM Training Loss', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
seed = "To be or not to be that is th"
print(f"Seed: '{seed}'")
print(f"\n--- Generated text (temperature=0.5, more conservative) ---")
print(char_model.generate(seed, length=150, temperature=0.5))
print(f"\n--- Generated text (temperature=1.0, more creative) ---")
print(char_model.generate(seed, length=150, temperature=1.0))

**Temperature** controls randomness:
- Low (0.2-0.5): more predictable, repetitive but coherent
- High (1.0-2.0): more creative, surprising but potentially nonsensical

With a tiny training corpus, the model can only learn local character patterns. Real language models train on billions of tokens.

---
## 6. Sequence Classification — Sentiment from Simple Sequences

Instead of generating text, we can use RNNs to *classify* sequences. We feed the whole sequence through the LSTM and use the final hidden state for classification.

In [ ]:
positive_words = ['good', 'great', 'excellent', 'wonderful', 'amazing', 'love', 'best', 'happy', 'fantastic', 'beautiful']
negative_words = ['bad', 'terrible', 'awful', 'horrible', 'hate', 'worst', 'sad', 'ugly', 'boring', 'poor']
neutral_words = ['the', 'a', 'is', 'it', 'was', 'very', 'really', 'so', 'quite', 'this', 'movie', 'film', 'book']

all_words = positive_words + negative_words + neutral_words
word_to_idx = {w: i+1 for i, w in enumerate(all_words)}
vocab_size_sent = len(word_to_idx) + 1

def make_sentence(label, length=5):
    words = []
    for _ in range(length):
        if np.random.random() < 0.4:
            words.append(np.random.choice(neutral_words))
        else:
            if label == 1:
                words.append(np.random.choice(positive_words))
            else:
                words.append(np.random.choice(negative_words))
    return words

np.random.seed(42)
sentences, labels = [], []
for _ in range(500):
    label = np.random.randint(0, 2)
    sent = make_sentence(label)
    sentences.append([word_to_idx[w] for w in sent])
    labels.append(label)

X_sent = torch.LongTensor(sentences)
y_sent = torch.FloatTensor(labels)

print(f"Sample positive: {make_sentence(1)}")
print(f"Sample negative: {make_sentence(0)}")
print(f"Dataset: {len(X_sent)} sentences, vocab size: {vocab_size_sent}")

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_size=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        return torch.sigmoid(self.fc(h_n[-1])).squeeze()

torch.manual_seed(42)
sent_model = SentimentLSTM(vocab_size_sent).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(sent_model.parameters(), lr=0.01)

split = 400
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_sent[:split], y_sent[:split]),
    batch_size=32, shuffle=True
)

losses = []
for epoch in range(30):
    sent_model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        pred = sent_model(X_batch)
        loss = criterion(pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(train_loader))

sent_model.eval()
with torch.no_grad():
    test_pred = sent_model(X_sent[split:].to(device))
    acc = ((test_pred.cpu() > 0.5).float() == y_sent[split:]).float().mean()

plt.plot(losses, color='#9b59b6', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title(f'Sentiment Classification — Test Acc: {acc:.1%}', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

---
## 7. GRU — A Simpler Alternative

GRUs (Gated Recurrent Units) are a simplified version of LSTMs:
- 2 gates instead of 3 (reset + update)
- No separate cell state
- Fewer parameters, often similar performance
- Drop-in replacement: just swap `nn.LSTM` → `nn.GRU`

In [ ]:
lstm_layer = nn.LSTM(input_size=10, hidden_size=20, batch_first=True)
gru_layer = nn.GRU(input_size=10, hidden_size=20, batch_first=True)

lstm_params = sum(p.numel() for p in lstm_layer.parameters())
gru_params = sum(p.numel() for p in gru_layer.parameters())

print(f"LSTM parameters: {lstm_params:,}  (4 weight matrices per gate)")
print(f"GRU parameters:  {gru_params:,}  (3 weight matrices per gate)")
print(f"GRU is {(1 - gru_params/lstm_params):.0%} fewer parameters")

x = torch.randn(3, 5, 10)
lstm_out, (h_lstm, c_lstm) = lstm_layer(x)
gru_out, h_gru = gru_layer(x)

print(f"\nLSTM output: {lstm_out.shape}, hidden: {h_lstm.shape}, cell: {c_lstm.shape}")
print(f"GRU output:  {gru_out.shape}, hidden: {h_gru.shape} (no cell state)")

---
## 8. Transformers Preview

RNNs have a fundamental limitation: they process sequences **one step at a time**. Step 50 must wait for steps 1-49 to finish. This means:
- No parallelization during training
- Long-range dependencies are still hard despite LSTM

**Transformers** (2017) solved both problems with **self-attention**:
- Process ALL positions simultaneously → massive parallelism
- Every position can directly attend to every other position → no distance limit
- The foundation for GPT, BERT, and every modern language model

```
RNN:          x1 → x2 → x3 → x4 → output  (sequential)
Transformer:  x1  x2  x3  x4  (parallel, all-to-all attention)
               ╲╱╲╱╲╱╲╱╲╱╲╱
              attention matrix
                    ↓
                 output
```

We'll cover Transformers in depth in Topic 04.

---
## Key Takeaways

1. **RNNs** maintain a hidden state that carries information across time steps
2. **Vanishing gradients** make basic RNNs unable to learn long-range dependencies
3. **LSTMs** solve this with forget/input/output gates and a cell state highway
4. **GRUs** are simpler (2 gates) with comparable performance
5. **Applications**: time series prediction, text generation, sequence classification
6. **Temperature** in generation controls creativity vs coherence
7. **Transformers** (next topic) are replacing RNNs for most sequence tasks due to parallelism and better long-range modeling